<font size=10>**DATA COLLECTION & QUALITY**</font> <a class="anchor" id='title'></a>

**Bachelor's in Data Science - NOVA IMS (26/27)**

**Course**: *Data Stream Mining — Practical Assignment I (PA48)*

**Project**: *Weather Windows for Outdoor Races — Angra do Heroísmo vs Braga*

**Group YY**
- Beatriz Marques 20231605
- Duarte Fernandes 20231619

**Question**: *Which city offers more reliable weather windows for recurring outdoor races?*

**Pipeline step**: This notebook reproduces **step 1** of `python src/practical_assignment.py --config config.yaml`: load the configuration and fetch the two hourly city streams. Quality reports and descriptive plots are additional inspection steps; they do not change the streams.

| Notebook | What it does |
|---|---|
| **01 — `01_data_collection_and_quality.ipynb`** | **Data collection and quality** |
| 02 — `02_target_and_features.ipynb` | Target, features and shared mask |
| 03 — `03_prequential_evaluation.ipynb` | Evaluation and audit outputs |
| 04 — `04_results_and_figures.ipynb` | Results, figures and audit checks |

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)

- [2. Configuration](#2)
    - [2.1 Assignment Card](#2_1)
    - [2.2 Cities, Period and Warm-up](#2_2)

- [3. Data Collection](#3)
    - [3.1 Angra do Heroísmo](#3_1)
    - [3.2 Braga](#3_2)

- [4. Data Quality](#4)
    - [4.1 Quality Report](#4_1)
    - [4.2 Descriptive Statistics](#4_2)
        - [Insights](#insights_1)

- [5. Stream Overview](#5)
    - [5.1 Daily Evolution](#5_1)
    - [5.2 Monthly profile](#5_2)
        - [Insights](#insights_2)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

Enable **automatic source reloads** and suppress warnings while running the notebook cells in order. Start Jupyter with this notebook in the repository’s `notebooks/` folder.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

Import the libraries and the documented **`src` API**. Add `../src` to `sys.path`; subsequent data and output paths are built from **`data_loader.ROOT`**, not the notebook working directory.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Get the absolute path of the source code folder
source_code_path = os.path.abspath('../src')

# Add the source code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from data_loader import ROOT, load_config, fetch_city, quality_report

# <font color='#BFD72F' size=6>**2. Configuration**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

Load **`config.yaml`** through `load_config(ROOT / "config.yaml")`. Read the period and warm-up from `cfg["period"]` and `cfg["warmup"]["n_hours"]`; resolve the configured raw-data directory against `ROOT`.

In [ ]:
cfg = load_config(ROOT / 'config.yaml')
period = cfg['period']
warm_n = cfg['warmup']['n_hours']
raw_dir = ROOT / cfg['paths']['raw_data']

## <font color='#BFD72F' size=5>2.1 Assignment Card</font> <a class="anchor" id="2_1"></a>

[Back to TOC](#toc)

Display **`cfg["assignment"]`**, including the assignment code, target and group, without changing the configuration.

In [ ]:
display(pd.Series(cfg['assignment'], name='value').to_frame())

## <font color='#BFD72F' size=5>2.2 Cities, Period and Warm-up</font> <a class="anchor" id="2_2"></a>

[Back to TOC](#toc)

Combine **`cfg["cities"]`**, **`cfg["period"]`** and the configured warm-up length into one small table.

In [ ]:
setup_rows = []
for key in ('city_a', 'city_b'):
    c = cfg['cities'][key]
    setup_rows.append({
        'city_key': key, 'city': c['name'],
        'latitude': c['latitude'], 'longitude': c['longitude'],
        'start_date': period['start_date'], 'end_date': period['end_date'],
        'timezone': period['timezone'], 'warmup_hours': warm_n,
    })
display(pd.DataFrame(setup_rows))
data = {}

# <font color='#BFD72F' size=6>**3. Data Collection**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

**`fetch_city`** caches each hourly stream as a CSV in the configured raw directory, normally repository `data/raw`. Its default **`use_cache=True`** reuses an available cache. To force a re-download, rerun the corresponding call with **`use_cache=False`**; the cache location is still `ROOT / cfg["paths"]["raw_data"]`.

## <font color='#BFD72F' size=5>3.1 Angra do Heroísmo</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

Fetch **`cfg["cities"]["city_a"]`** with `fetch_city`, using the configured coordinates, period and timezone. Inspect the shape, first rows and column types.

In [ ]:
c = cfg['cities']['city_a']
data['city_a'] = fetch_city(
    c['name'], c['latitude'], c['longitude'],
    period['start_date'], period['end_date'], period['timezone'],
    raw_dir=raw_dir,
)
print(f"{c['name']}: shape={data['city_a'].shape}")
display(data['city_a'].head())
display(data['city_a'].dtypes.rename('dtype').to_frame())

## <font color='#BFD72F' size=5>3.2 Braga</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

Fetch **`cfg["cities"]["city_b"]`** with `fetch_city`, using the configured coordinates, period and timezone. Inspect the shape, first rows and column types.

In [ ]:
c = cfg['cities']['city_b']
data['city_b'] = fetch_city(
    c['name'], c['latitude'], c['longitude'],
    period['start_date'], period['end_date'], period['timezone'],
    raw_dir=raw_dir,
)
print(f"{c['name']}: shape={data['city_b'].shape}")
display(data['city_b'].head())
display(data['city_b'].dtypes.rename('dtype').to_frame())

# <font color='#BFD72F' size=6>**4. Data Quality**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>4.1 Quality Report</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

Run **`quality_report(df)`** for both streams. Flatten its per-column **`missing`** dictionary into `missing_<column>` fields; keep one report row per city. No cleaning or row removal is performed here.

In [ ]:
quality_rows = []
for key, df in data.items():
    report = quality_report(df)
    row = {'city': cfg['cities'][key]['name']}
    row.update({k: v for k, v in report.items() if k != 'missing'})
    row.update({f'missing_{column}': count for column, count in report['missing'].items()})
    quality_rows.append(row)
display(pd.DataFrame(quality_rows))

## <font color='#BFD72F' size=5>4.2 Descriptive Statistics</font> <a class="anchor" id="4_2"></a>

[Back to TOC](#toc)

Show **`DataFrame.describe()`** for the four measured variables, with the two city summaries side by side. These summaries are descriptive only.

In [ ]:
variables = ['temperature', 'relative_humidity', 'precipitation', 'wind_speed']
statistics = pd.concat(
    {cfg['cities'][key]['name']: df[variables].describe() for key, df in data.items()},
    axis=1,
)
display(statistics)

#### **Insights** <a class="anchor" id='insights_1'></a>
*To be written by the group.*

# <font color='#BFD72F' size=6>**5. Stream Overview**</font> <a class="anchor" id="5"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>5.1 Daily Evolution</font> <a class="anchor" id="5_1"></a>

[Back to TOC](#toc)

Resample each stream by **calendar day**. Use daily **means** for temperature, relative humidity and wind speed, and daily **sums** for precipitation. This display does not replace the hourly data used by the pipeline.

In [ ]:
labels = {
    'temperature': 'Temperature', 'relative_humidity': 'Relative humidity',
    'precipitation': 'Precipitation', 'wind_speed': 'Wind speed',
}
aggregations = {
    'temperature': 'mean', 'relative_humidity': 'mean',
    'precipitation': 'sum', 'wind_speed': 'mean',
}
fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex='col', squeeze=False)
for j, key in enumerate(('city_a', 'city_b')):
    hourly = data[key].set_index('time')[variables]
    daily = hourly.resample('D').agg(aggregations)
    for i, variable in enumerate(variables):
        ax = axes[i, j]
        ax.plot(daily.index, daily[variable], linewidth=0.9)
        statistic = 'sum' if variable == 'precipitation' else 'mean'
        ax.set_title(f"{cfg['cities'][key]['name']} — {labels[variable]} (daily {statistic})")
        ax.set_ylabel(f'Daily {statistic}')
        ax.set_xlabel('Date')
        ax.grid(alpha=0.3)
fig.suptitle('Daily evolution of the hourly weather streams', y=1.01)
fig.tight_layout()
plt.show()

## <font color='#BFD72F' size=5>5.2 Monthly profile</font> <a class="anchor" id="5_2"></a>

[Back to TOC](#toc)

Group the **original hourly observations** by calendar month and compute each variable’s mean for each city. Month numbers pool the same calendar month across years; precipitation here is the mean hourly amount, not a monthly total.

In [ ]:
monthly_profiles = {}
for key, df in data.items():
    monthly_profiles[cfg['cities'][key]['name']] = (
        df.groupby(df['time'].dt.month)[variables].mean().reindex(range(1, 13))
    )
display(pd.concat(monthly_profiles, axis=1).rename_axis('calendar_month'))
fig, axes = plt.subplots(2, 2, figsize=(14, 8), squeeze=False)
for ax, variable in zip(axes.flat, variables):
    for city, profile in monthly_profiles.items():
        ax.plot(profile.index, profile[variable], marker='o', label=city)
    ax.set_title(f'{labels[variable]} — calendar-month profile')
    ax.set_xlabel('Calendar month')
    ax.set_ylabel('Mean hourly value')
    ax.set_xticks(range(1, 13))
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

#### **Insights** <a class="anchor" id='insights_2'></a>
*To be written by the group.*